# DTAT357. deeptrack.elementwise

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT357_elementwise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the module [deeptrack.elementwise](../../deeptrack/elementwise.py).

## 1. What is `elementwise.py`?

The `elementwise.py` module introduces utility functions which lets the user apply NumPy or PyTorch functions to `Feature` objects elementwise, that is, element by element.

Supported categories include:
- Trigonometric functions
- Hyperbolic functions
- Rounding operations
- Exponents and logarithms
- Complex number operations (real, imag, conj, angle)
- Miscellaneous (e.g., abs, square, sqrt, sign)

The key roles of `elementwise.py` are:

- **Backend-Agnostic Mathematical Transformations:**
  The module provides a unified interface to apply mathematical functions
  (e.g., `sin`, `exp`, `sqrt`, `abs`) elementwise across both NumPy and 
  PyTorch backends. This ensures compatibility without requiring users to 
  manage backend-specific logic.

- **Composable Elementwise Features:**
  Each function is wrapped in a `Feature` class, making it fully compatible 
  with DeepTrack2 pipelines. Users can plug these features directly into 
  image-processing or data-generation workflows.

- **Factory-Based Extensibility:**
  The `create_elementwise_class()` function allows dynamically generating 
  new elementwise features for backend functions, simplifying extension and 
  ensuring consistency across the API.

- **Automatic Support for Static and Dynamic Inputs:**
  Elementwise features automatically handle both static inputs (e.g., 
  `np.array`) and dynamic `Feature`-based inputs, preserving full support 
  for lazy evaluation, parameterization, and randomization.

- **Robust Compatibility Layer for Edge Cases:**
  For functions like `ceil`, `floor`, `sign`, or `imag` that are not safely 
  handled by the `array-api-compat` layer, the module provides manually 
  defined classes with backend-specific dispatching to ensure stability and 
  correctness.

- **Operator Chaining and Feature Composition:**
  Elementwise features support chaining via the `>>` operator or nesting in 
  constructors, allowing concise and readable expression of transformation 
  pipelines.

## 2. Using Elementwise Features

### 2.1. Using with NumPy Array with Direct Input

In [2]:
from deeptrack.elementwise import Abs
import numpy as np

array = np.array([-1.0, 0.0, 2.5])
result = Abs()(array)
result

array([1. , 0. , 2.5])

### 2.2. Using with PyTorch Tensor with Direct Input

In [3]:
import torch

tensor = torch.tensor([-1.0, 0.0, 2.5])
result = Abs()(tensor)
result

tensor([1.0000, 0.0000, 2.5000])

### 2.3. Using in a NumPy Pipeline

You can also wrap the NumPy array using `dt.Value` and apply the feature
as part of a pipeline using the `>>` operator.

In [4]:
import deeptrack as dt

value = dt.Value(value=np.array([-3.0, 0.0, 3.0]))
pipeline = value >> Abs()
result = pipeline()
result

array([3., 0., 3.])

Or equivalently:

In [5]:
value = dt.Value(value=np.array([-3.0, 0.0, 3.0]))
pipeline = Abs(value)
result = pipeline()
result

array([3., 0., 3.])

### 2.4. Using in a PyTorch Pipeline

Same idea using a PyTorch tensor.

In [6]:
value = dt.Value(value=torch.tensor([-3.0, 0.0, 3.0]))
pipeline = value >> Abs()
result = pipeline()
result

tensor([3., 0., 3.])

Or equivalently:

In [7]:
value = dt.Value(value=torch.tensor([-3.0, 0.0, 3.0]))
pipeline = Abs(value)
result = pipeline()
result

tensor([3., 0., 3.])

## 3. Creating an Elementwise Feature

### 3.1. Creating an Elementwise Feature Using the Factory Function

The most convenient way to define new elementwise operations in DeepTrack2
is through the `create_elementwise_class` factory function.

This function takes three arguments:
- `name`: A string defining the name of the class.
- `function`: A callable (usually from `xp`, `np`, or `torch`) to be applied
  elementwise.
- `docstring`: A documentation string (optional but recommended).

The factory function dynamically generates a subclass of `ElementwiseFeature`
that wraps the provided function and enables it to be used in any DeepTrack
pipeline. This avoids the need to manually define the `.__init__()` method or
dispatch between backends like NumPy and PyTorch.

Below we define a custom elementwise feature called `Abs` that applies the
absolute value function to its input.

In [8]:
from deeptrack.backend import xp
from deeptrack.elementwise import create_elementwise_class

Abs = create_elementwise_class(
    name="Abs",
    function=xp.abs,
    docstring="Elementwise absolute value feature.",
)

To check that this class behaves as expected, create an instance of the Abs feature:

In [9]:
abs_feature = Abs()

Evaluate the feature with NumPy input:

In [10]:
numpy_input = np.array([-3.0, 0.0, 2.5])
numpy_output = abs_feature(numpy_input)
print("NumPy input:", numpy_input)
print("NumPy output:", numpy_output)

NumPy input: [-3.   0.   2.5]
NumPy output: [3.  0.  2.5]


Evaluate the same feature with PyTorch input:

In [11]:
torch_input = torch.tensor([-3.0, 0.0, 2.5])
torch_output = abs_feature(torch_input)
print("PyTorch input:", torch_input)
print("PyTorch output:", torch_output)

PyTorch input: tensor([-3.0000,  0.0000,  2.5000])
PyTorch output: tensor([3.0000, 0.0000, 2.5000])


Use Abs in a DeepTrack pipeline with NumPy:

In [12]:
value_np = dt.Value(value=np.array([-3.0, 0.0, 2.5]))
pipeline_np = value_np >> Abs()
print("Pipeline output (NumPy):", pipeline_np())

Pipeline output (NumPy): [3.  0.  2.5]


Use Abs in a DeepTrack pipeline with PyTorch:

In [13]:
value_torch = dt.Value(value=torch.tensor([-3.0, 0.0, 2.5]))
pipeline_torch = value_torch >> Abs()
print("Pipeline output (PyTorch):", pipeline_torch())

Pipeline output (PyTorch): tensor([3.0000, 0.0000, 2.5000])


### 3.2. Creating an Elementwise Feature Inheriting from `ElementWise`

Some mathematical operations, such as `ceil`, require **manual subclassing**
from `ElementwiseFeature`. This is necessary when the corresponding
`array-api-compat` function (e.g., `xp.ceil`) does not safely handle all
backends.

In particular, calling `xp.ceil` on a `torch.Tensor` fails because it internally
uses `xp.issubdtype`, which cannot interpret PyTorch dtypes like
`torch.float32`. To avoid this issue, we subclass `ElementwiseFeature` and
manually dispatch to the appropriate backend-specific function.

The example below demonstrates how to implement a `Ceil` feature:

In [14]:
from deeptrack import Feature, TORCH_AVAILABLE
from deeptrack.elementwise import ElementwiseFeature
import numpy as np

if TORCH_AVAILABLE:
    import torch


class Ceil(ElementwiseFeature):
    """Apply the ceiling function elementwise for NumPy and PyTorch arrays.

    This implementation dispatches to `torch.ceil` for PyTorch tensors and
    `np.ceil` for NumPy arrays, ensuring compatibility across backends.

    """

    def __init__(self, feature, **kwargs):
        super().__init__(
            function=self._ceil_dispatch,
            feature=feature,
            **kwargs,
        )

    @staticmethod
    def _ceil_dispatch(x):
        """Dispatch the ceiling function depending on the backend.

        Returns
        -------
        array or tensor with elementwise ceil applied.

        """

        if TORCH_AVAILABLE and isinstance(x, torch.Tensor):
            return torch.ceil(x)

        return np.ceil(x)

This implementation overrides the `.__init__()` method to pass a custom dispatch function _ceil_dispatch to ElementwiseFeature. The dispatch function checks whether the input is a `torch.Tensor` or a `np.ndarray` and applies the appropriate backend function.

This approach ensures correct operation on both NumPy and PyTorch backends, and avoids failures that would arise from using the factory function.